In [1]:
import os
import duckdb
import pandas as pd

In [3]:
con = duckdb.connect()
con.execute(open("/workspace/sql/00_attach.sql").read())
print("Connected.")

Connected.


In [4]:
con.execute("""
    CREATE TABLE silver.coco_annotations AS
    SELECT DISTINCT ON (bbox_id)
        image_uri,
        image_id,
        width,
        height,
        bbox_id,
        category,
        CAST(regexp_extract(bbox, '\[([0-9.]+),', 1) AS DOUBLE) AS bbox_x,
        CAST(regexp_extract(bbox, '\[[0-9.]+,\s*([0-9.]+),', 1) AS DOUBLE) AS bbox_y,
        CAST(regexp_extract(bbox, '\[[0-9.]+,\s*[0-9.]+,\s*([0-9.]+),', 1) AS DOUBLE) AS bbox_w,
        CAST(regexp_extract(bbox, '\[[0-9.]+,\s*[0-9.]+,\s*[0-9.]+,\s*([0-9.]+)\]', 1) AS DOUBLE) AS bbox_h,
        area,
        'val' AS split
    FROM raw.coco_annotations
    WHERE area > 0
""")
print("silver.coco_annotations created.")

<>:10: SyntaxWarning: invalid escape sequence '\['
<>:10: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_27288/3858210126.py:10: SyntaxWarning: invalid escape sequence '\['
  CAST(regexp_extract(bbox, '\[([0-9.]+),', 1) AS DOUBLE) AS bbox_x,


silver.coco_annotations created.


In [5]:
con.execute("""
    CREATE TABLE silver.visdrone_annotations AS
    SELECT DISTINCT ON (clip_uri, frame_id, target_id)
        clip_uri,
        sequence,
        fragment_id,
        frame_id,
        target_id,
        x, y, w, h,
        (w * h) AS area,
        category,
        truncation,
        occlusion
    FROM raw.visdrone_annotations
    WHERE category != 'ignored'
""")
print("silver.visdrone_annotations created.")

silver.visdrone_annotations created.


In [6]:
con.execute("""
    CREATE TABLE silver.visdrone_fragments AS
    SELECT * FROM raw.visdrone_fragments
""")
print("silver.visdrone_fragments created.")

silver.visdrone_fragments created.


In [7]:
print(con.sql("FROM ducklake_snapshots('lake')").df())

   snapshot_id                    snapshot_time  schema_version  \
0            0 2026-06-25 19:47:58.592626+00:00               0   
1            1 2026-06-25 19:47:58.638004+00:00               1   
2            2 2026-06-25 19:47:58.647187+00:00               2   
3            3 2026-06-25 19:47:58.669349+00:00               3   
4            4 2026-06-25 22:18:44.460503+00:00               4   
5            5 2026-06-25 23:13:50.427815+00:00               5   
6            6 2026-06-25 23:13:50.502705+00:00               6   
7            7 2026-06-26 00:31:41.238716+00:00               7   
8            8 2026-06-26 00:31:56.698735+00:00               8   
9            9 2026-06-26 00:32:06.396628+00:00               9   

                                             changes author commit_message  \
0                      {'schemas_created': ['main']}   None           None   
1                       {'schemas_created': ['raw']}   None           None   
2                    {'schem

In [8]:
print(con.sql("SELECT COUNT(*) FROM silver.coco_annotations").df())
print(con.sql("SELECT COUNT(*) FROM silver.visdrone_annotations").df())
print(con.sql("SELECT COUNT(*) FROM silver.visdrone_fragments").df())
con.sql("SELECT image_uri, category, bbox_x, bbox_y, bbox_w, bbox_h, split FROM silver.coco_annotations LIMIT 5").df()

   count_star()
0         36781
   count_star()
0        114133
   count_star()
0            99


,image_uri,category,bbox_x,bbox_y,bbox_w,bbox_h,split
0,s3://lakehouse/assets/coco/images/139.jpg,75,336.79,199.50,346.52,216.23,val
1,s3://lakehouse/assets/coco/images/632.jpg,73,487.51,199.33,494.99,227.38,val
2,s3://lakehouse/assets/coco/images/632.jpg,73,497.39,55.43,501.47,82.79,val
3,s3://lakehouse/assets/coco/images/872.jpg,32,408.03,172.04,427.41,188.57,val
4,s3://lakehouse/assets/coco/images/885.jpg,0,277.31,189.99,417.40,398.21,val


In [9]:
con.close()